## Accumulation Error Factor Analysis
Exploration of errors during accumulation for Geostats term project. 

In [1]:
# Site and Model Run
site = 'Quemazon'
namelist = 'ghelaps'
forcing_str = '01011010'
version = '0008'
platform = 'laptop'  # desktop, laptop, alpine

In [2]:
# Load environmental variables if on Alpine, otherwise they should be stored
import os

# Configure environmental variables
if platform == 'alpine':
    os.environ['OUTPUT_PATH'] = r'/projects/ccox1@xsede.org/lm4px_modelling/output' #Alpine
    os.environ['SNOTEL_PATH'] = r'/projects/ccox1@xsede.org/data/snotel/'
    os.environ['FORCING_PATH'] = r'/projects/ccox1@xsede.org/lm4px_modelling/forcing/'
elif platform == 'laptop':
    os.environ['OUTPUT_PATH'] = r'/home/chriscox/Data/output/'
    os.environ['FORCING_PATH'] = r'/home/chriscox/Data/input/forcing/'
    os.environ['SNOTEL_PATH'] = r'/home/chriscox/Data/snotel/'

In [ ]:
%matplotlib ipympl
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
#from model_datasets.forcing import LM4Forcing
from model_datasets.output import LM4ModelOutput 
from buildforcing.datasets import PNNLSnotel

Load datasets

In [ ]:
model = LM4ModelOutput(site=site, forcing_string=forcing_str, namelist=namelist, version=version, spinup=False)

In [ ]:
daily_vars = model.get_variables(diag='land_daily')
monthly_vars = model.get_variables(diag='land_month')

In [ ]:
snotel = PNNLSnotel(site, os.environ['SNOTEL_PATH'])
snotel.load_data()

### Calculate daily error

In [ ]:
snotel.data.head()

In [ ]:
model_snow = model.get_data(['snow'],'land_daily')

# Create combined dataframe with model snow
combined_data = pd.DataFrame(model_snow['snow'])

Since all timezones in the western US don't exceed UTC-9, we can safely remove timezones for merging purposes.

In [ ]:
# Combine with snotel data
snotel_swe = snotel.data['swe_mm']

# Remove timezone from index
snotel_swe.index = snotel_swe.index.tz_localize(None)

# Shift snotel data by 12 hours to match model output time
snotel_swe.index = snotel_swe.index + pd.Timedelta(hours=12)

combined_data = pd.merge(combined_data, snotel_swe, how='left', left_index=True, right_index=True)

In [ ]:
# Calculate the daily error between model and snotel
combined_data['error'] = combined_data['snow'] - combined_data['swe_mm']

In [ ]:
# Plot snow and error in one figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# First subplot: Model Snow vs SNOTEL Snow
ax1.plot(combined_data.index, combined_data['snow'], label=f'Model Snow {site}', color='orange')
ax1.plot(combined_data.index, combined_data['swe_mm'], label=f'SNOTEL Snow {site}', color='blue')
ax1.set_ylabel('Snow (mm)')
ax1.legend()

# Second subplot: Error
ax2.plot(combined_data.index, combined_data['error'], label=f'Error {site}', color='cyan')
ax2.set_ylabel('Error (mm)')
ax2.set_xlabel('Date')
ax2.legend()